# Detailed Numerical Evaluation of Formulas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special
from pathlib import Path
from compute_H import H_pq, H_pq_decaying, kernel_bracket, g_decaying
OUT=Path('../outputs'); OUT.mkdir(exist_ok=True)
M,R,k_0,C_k,epsilon=0.1,1e4,1.0,1.6,1e-3

## Cell Group 2: Projection Tensor Operations

In [ ]:
def projection_tensor(khat):
    khat=np.asarray(khat,float); khat=khat/np.linalg.norm(khat)
    return np.eye(3)-np.outer(khat,khat)
angles=np.linspace(0,np.pi,200)
vals=1+np.cos(angles)**2
plt.figure(figsize=(5,3)); plt.plot(angles,vals); plt.xlabel('angle'); plt.ylabel('1+cos^2'); plt.tight_layout(); plt.savefig(OUT/'geometric_factor_vs_angle.png',dpi=160); plt.close()

## Cell Group 3: Dimension Analysis & Scaling Relations

In [ ]:
ks=np.logspace(0,3,200)
A=C_k*epsilon**(2/3)*ks**(-11/3)/(4*np.pi)
Ek=C_k*epsilon**(2/3)*ks**(-5/3)
eta=epsilon**(1/3)*ks**(2/3)/np.sqrt(2*np.pi)
tau=np.linspace(-6,6,400)
f0=np.exp(-(np.pi/4)*(eta[40]**2)*(tau**2))
omega=np.linspace(-8,8,400)
ft_num=np.array([np.trapz(f0*np.exp(1j*w*tau),tau) for w in omega]).real
plt.figure(figsize=(6,4)); plt.loglog(ks,Ek,label='E_k'); plt.loglog(ks,A,label='A(k)'); plt.legend(); plt.tight_layout(); plt.savefig(OUT/'Ek_Ak_scaling.png',dpi=160); plt.close()

## Cell Group 4: Stationary Spectrum H(p,q)

In [ ]:
ps=np.logspace(-2,0.8,12); qs=np.logspace(-2,0.8,12)
Hgrid=np.zeros((len(qs),len(ps)))
for i,q in enumerate(qs):
    for j,p in enumerate(ps):
        Hgrid[i,j]=H_pq(p,q,M=M,R=R,k0=k_0,epsabs=5e-4,epsrel=3e-3)
np.save(OUT/'Hgrid_from_compute_detailed.npy', {'ps':ps,'qs':qs,'H':Hgrid})
plt.figure(figsize=(6,4)); plt.pcolormesh(ps,qs,Hgrid,shading='auto'); plt.xscale('log'); plt.yscale('log'); plt.colorbar(label='H'); plt.tight_layout(); plt.savefig(OUT/'Hgrid_stationary_compute_detailed.png',dpi=160); plt.close()

## Cell Group 5: Split Form with I_{alpha}

In [ ]:
def I_alpha(q,Om,Om1,alpha):
    a,b=abs(1-q),1+q
    f=lambda s: s**(-10/3+alpha)*np.exp(-2*(Om-Om1)**2*s**(-4/3))
    return integrate.quad(f,a,b,limit=200,epsabs=1e-7,epsrel=1e-6)[0]
def H_split_point(q,Om,Om1=0.4):
    c1=(27/6)-(1/(6*q**2)); c2=(1/(12*q**2))+(q**2/12)-1/6; c3=1/(12*q**2)
    return c1*I_alpha(q,Om,Om1,1)+c2*I_alpha(q,Om,Om1,-1)+c3*I_alpha(q,Om,Om1,3)
q0,Om0=0.7,0.8
print('split value',H_split_point(q0,Om0))

## Cell Group 6: Special case p->0

In [ ]:
small_ps=np.array([1e-3,2e-3,5e-3,1e-2])
vals=np.array([H_pq(p,0.3,M=M,R=R,k0=k_0,epsabs=5e-4,epsrel=3e-3) for p in small_ps])
print(np.c_[small_ps,vals])

## Cell Group 7: Decaying Spectrum Model

In [ ]:
z=np.linspace(-20,20,600); gz=g_decaying(z)
plt.figure(figsize=(6,4)); plt.plot(z,gz.real,label='Re'); plt.plot(z,gz.imag,label='Im'); plt.legend(); plt.tight_layout(); plt.savefig(OUT/'g_decaying_real_imag.png',dpi=160); plt.close()
conv=lambda q,x,y: integrate.quad(lambda q1:(g_decaying(q1*np.sqrt(x)/M)*g_decaying((q-q1)*np.sqrt(y)/M)).real,-np.inf,np.inf,limit=150,epsabs=1e-5,epsrel=1e-4)[0]
print('conv sample',conv(0.8,0.6,1.2))

In [ ]:
Hdec=np.zeros_like(Hgrid)
for i,q in enumerate(qs):
    for j,p in enumerate(ps):
        Hdec[i,j]=H_pq_decaying(p,q,M=M,R=R,k0=k_0,epsabs=2e-3,epsrel=1e-2)
plt.figure(figsize=(6,4)); plt.pcolormesh(ps,qs,Hdec,shading='auto'); plt.xscale('log'); plt.yscale('log'); plt.colorbar(label='H dec'); plt.tight_layout(); plt.savefig(OUT/'Hgrid_decaying_compute_detailed.png',dpi=160); plt.close()

## Cell Group 8: Gogoberidze 2007 Appendix A Form

In [ ]:
def gogoberidze_integrand_u(u,k,k1,w,eps=epsilon):
    A=2*(k1**(-4/3)+u**(-4/3)); z=w/(eps**(1/3)*k**(2/3)+1e-15)
    bracket=27-k**2/k1**2-k**2/u**2+0.5*k**4/(k1**2*u**2)+0.5*k1**2/u**2+0.5*u**2/k1**2
    return u**(-10/3)*(k1**(-4/3)+u**(-4/3))**(-0.5)*bracket*np.exp(-(z**2)/A)*special.erfc(-z/np.sqrt(A))
def H_gogo(k,w,k0=1.0,kd=30.0):
    pref=epsilon/(24*np.pi*(2*np.pi)**1.5*k)
    def inner_k1(k1):
        umin=max(abs(k1-k),k0); umax=min(k1+k,kd)
        if umin>=umax: return 0.0
        return k1**(-10/3)*integrate.quad(gogoberidze_integrand_u,umin,umax,args=(k,k1,w),limit=150,epsabs=1e-6,epsrel=1e-5)[0]
    return pref*integrate.quad(inner_k1,k0,kd,limit=120,epsabs=1e-6,epsrel=1e-5)[0]
print('H_gogo sample',H_gogo(1.2,0.7))

## Cell Group 9: Consistency Checks

In [ ]:
pt={'p':0.4,'q':0.6}
h_direct=H_pq(pt['p'],pt['q'],M=M,R=R,k0=k_0,epsabs=5e-4,epsrel=3e-3)
h_split=H_split_point(pt['q'],pt['q']/M,Om1=0.4)
print('direct',h_direct,'split proxy',h_split)
ks=np.logspace(0,2,15); slope=np.polyfit(np.log(ks),np.log(ks**(-5)),1)[0]; print('k^-5 slope',slope)

## Cell Group 10: Visualization & Results

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,4),sharey=True)
pcm=ax[0].pcolormesh(ps,qs,Hgrid,shading='auto'); ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_title('stationary')
ax[1].pcolormesh(ps,qs,Hdec,shading='auto'); ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].set_title('decaying')
for a in ax: a.set_xlabel('p'); a.set_ylabel('q')
fig.colorbar(pcm,ax=ax.ravel().tolist(),label='H')
fig.tight_layout(); fig.savefig(OUT/'stationary_vs_decaying_side_by_side.png',dpi=160); plt.close(fig)